# Concorde — exploration et démonstration locale

Ce notebook inventorie les projets du serveur local, peut lancer une analyse complète du projet choisi, valide le JSON généré et vérifie les exports. Aucun contenu du défi ni aucune sortie calculée ne doit être enregistré dans le notebook ou poussé vers un service externe.

**Prérequis :** démarrez le serveur avec `start-concorde.ps1`. Le pipeline complet peut prendre longtemps et utiliser le GPU. La cellule d’analyse reste désactivée jusqu’à ce que `RUN_FULL_PIPELINE` soit mis à `True`.

In [ ]:
from pathlib import Path
import json, sys, time
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from l2c_app.config import STORE
from l2c_app.models import Information
from l2c_app.ocr import hardware
BASE_URL = "http://127.0.0.1:8765"
def api_json(path, method="GET", value=None):
    body = None if value is None else json.dumps(value).encode("utf-8")
    headers = {"Accept": "application/json"}
    if body is not None: headers.update({"Content-Type": "application/json", "X-Concorde-Request": "1"})
    request = Request(BASE_URL + path, data=body, headers=headers, method=method)
    with urlopen(request, timeout=10) as response: return json.loads(response.read().decode("utf-8"))
overview = api_json("/api/overview")
hardware()

## Inventaire local

Choisir le projet uniquement parmi les PDF présents dans le catalogue. Les pages sans texte natif peuvent être des dessins vectoriels; elles ne sont pas comptées comme pages vides.

In [ ]:
projects = overview['projects']
[(p['id'], p['name'], p['plan_pages'], p['atelier_pages'], p['pages']) for p in projects]

## Pipeline complet sur un projet

Choisir un `PROJECT_ID` dans l’inventaire puis mettre `RUN_FULL_PIPELINE = True`. Le profil `complete` parcourt toutes les pages disponibles et crée les exports dans le dossier local `STORE/runs/<run_id>/`. Le serveur local sérialise les analyses et utilise le cache de pages. Ne pas démarrer une deuxième analyse en parallèle.

In [ ]:
PROJECT_ID = ''  # identifiant exact affiché dans l’inventaire
RUN_FULL_PIPELINE = False
if RUN_FULL_PIPELINE:
    project_ids = {p['id'] for p in overview['projects']}
    if PROJECT_ID not in project_ids:
        raise ValueError("Choisir un PROJECT_ID présent dans l’inventaire local.")
    job = api_json(f"/api/projects/{PROJECT_ID}/analyse", method="POST",
                   value={"profile": "complete", "max_ocr_pages": 1000})
    print("Run", job["id"], "démarré")
    last_print = 0
    while True:
        time.sleep(1)
        current = api_json(f"/api/jobs/{job['id']}")
        if time.monotonic() - last_print >= 5 or current["status"] not in ("queued", "running"):
            print(f"{current['current']}/{current['total']} pages — {current['status']}")
            last_print = time.monotonic()
        if current["status"] not in ("queued", "running"):
            break
    if current["status"] != "completed":
        raise RuntimeError(current.get("message", "Analyse incomplète"))
    run_path = STORE / "runs" / job["id"] / "run.json"
    run = json.loads(run_path.read_text(encoding="utf-8"))
    export_path = STORE / "runs" / job["id"] / "informations.json"
    exported = json.loads(export_path.read_text(encoding="utf-8"))
    for value in exported: Information.model_validate(value)
    import pymupdf as fitz
    report_path = STORE / "runs" / job["id"] / "rapport.pdf"
    with fitz.open(report_path) as report: report_pages = len(report)
    print("Couverture:", run["statistics"])
    print("Portée:", run["scope"], "— pages traitées:", run["statistics"]["pages_processed"], "/", run["statistics"]["pages_total"])
    print("JSON valide:", len(exported), "annotations; rapport:", report_pages, "pages")
    print("Exports locaux:", export_path, STORE / "runs" / job["id"] / "comparaisons.json", report_path)
else:
    print("Analyse désactivée. Choisir le projet et mettre RUN_FULL_PIPELINE à True pour lancer le profil complet.")

## Interpréter et valider

- Une exécution complète prouve la couverture des pages, pas l’exactitude de toutes les annotations. Consulter les erreurs et méthodes de lecture page par page.
- Vérifier des coordonnées sur les rotations 0°, 90°, 180° et 270° : elles doivent désigner le centre de l’annotation en points PDF depuis le coin supérieur gauche.
- Séparer CER/WER OCR, exactitude d’extraction, qualité d’appariement et précision/rappel des écarts.
- Les paires et non-conformités doivent être adjudiquées par un ingénieur. Garder toutes les pages/révisions d’un même projet ou dessin dans le même groupe; ne jamais répartir des tuiles corrélées entre les splits.
- Les quatre projets fournis ont déjà été explorés; ne pas les présenter comme un holdout complètement vierge. Le projet d’évaluation du jury n’est pas disponible avant la démo.